# Tutorial 3: Spatial Neighborhood Analysis with Squidpy

**Learning Objectives:**
- Build spatial neighborhood graphs from IMC data
- Quantify cell type co-occurrence patterns
- Perform neighborhood enrichment analysis
- Interpret spatial organization biologically

**Duration:** ~75 minutes

---

## Background

In the tumor microenvironment, cell-cell interactions drive immune responses, tumor progression, and treatment outcomes. **Spatial neighborhood analysis** quantifies which cell types tend to be near each other more (or less) than expected by chance.

### Key Concepts:

1. **Spatial Graph**: Connect cells that are spatial neighbors (e.g., within 20 μm)
2. **Neighborhood Enrichment**: Test if cell type A is found near cell type B more than expected
3. **Co-occurrence Score**: Quantitative measure of spatial association between cell types

**Biological Relevance:**
- CD8 T cells near tumor cells → potential immune response
- Macrophages near blood vessels → tissue remodeling
- B cells clustering together → tertiary lymphoid structures

---

## Dataset

We use the same IMC breast cancer dataset from previous tutorials:

- **Source**: Jackson Fischer Lab, University of Zurich
- **Technology**: Imaging Mass Cytometry (IMC)
- **Data**: 253,433 single cells across 132 images
- **Markers**: 40 protein markers
- **Cell types**: 14 annotated types (tumor, immune, stromal)

**Download location**: [Google Drive Data Folder](https://drive.google.com/drive/folders/1pLrAb0Hy6kudQ-BHZ1w_afq18Z9eu_RE)

Place files in `data/train_adata.h5ad` and `data/test_adata.h5ad`.

## 1. Setup and Load Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import anndata
import squidpy as sq

# Set plot style
plt.style.use('default')
sns.set_palette('tab10')

print(f"squidpy version: {sq.__version__}")
print(f"anndata version: {anndata.__version__}")

In [ ]:
# Load the dataset
DATA_PATH = 'data/train_adata.h5ad'
adata = anndata.read_h5ad(DATA_PATH)

print(f"Loaded: {adata.n_obs} cells, {adata.n_vars} markers")
print(f"Images: {adata.obs['image'].nunique()}")
print(f"Cell types: {adata.obs['celltypes'].nunique()}")

## 2. Understanding Spatial Graphs

A **spatial graph** connects cells based on their physical proximity. Squidpy uses the spatial coordinates stored in `adata.obsm['spatial']` to build this graph.

### Graph Construction Methods:

1. **KNN (k-nearest neighbors)**: Connect each cell to its k closest neighbors
2. **Radius**: Connect cells within a fixed distance threshold
3. **Delaunay**: Connect cells based on Delaunay triangulation

For IMC data, **radius-based graphs** are most interpretable because neighbor relationships depend on actual biological distances, not arbitrary k values.

### Exercise 2.1: Build a Spatial Graph for One Image

Let's start with a single image to understand the concept.

In [ ]:
# Select one image for demonstration
IMAGE_ID = 1
adata_single = adata[adata.obs['image'] == IMAGE_ID].copy()

print(f"Image {IMAGE_ID}: {adata_single.n_obs} cells")
print(f"Spatial coordinates shape: {adata_single.obsm['spatial'].shape}")

In [ ]:
# Build spatial graph with 20 μm radius
# coord_type='generic' means we use custom coordinates (not grid-based)
sq.gr.spatial_neighbors(
    adata_single,
    coord_type='generic',
    radius=20,  # Distance threshold in μm
    spatial_key='spatial'
)

print(f"Graph stored in: adata_single.obsp['spatial_connectivities']")
print(f"Graph shape: {adata_single.obsp['spatial_connectivities'].shape}")
print(f"Number of edges: {adata_single.obsp['spatial_connectivities'].nnz}")

### Exercise 2.2: Visualize the Spatial Graph

Let's visualize a small region to see how cells are connected.

In [ ]:
# Select a small region (cells 0-200) for visualization
subset = adata_single[:200]

fig, ax = plt.subplots(1, 1, figsize=(10, 10))

# Plot spatial graph
sq.pl.spatial_scatter(
    subset,
    color='celltypes',
    connectivity_key='spatial_connectivities',
    edges_width=0.5,
    edges_color='gray',
    size=50,
    ax=ax
)

ax.set_title(f'Spatial Graph (radius=20 μm)\nGray lines connect neighbors', fontsize=14)
plt.tight_layout()
plt.show()

**Interpretation:**
- Each dot is a cell, colored by cell type
- Gray edges connect cells within 20 μm of each other
- Dense regions have many connections (neighborhoods)
- Sparse regions have fewer connections

### Question 2.1: 
**What happens if you change the radius parameter to 10 μm or 40 μm? How does this affect the number of edges?**

*Hint: Try rerunning the graph construction with different radius values.*

In [ ]:
# Your exploration here

## 3. Neighborhood Enrichment Analysis

Now we'll test which cell types are spatially associated with each other.

### The Test:

For each pair of cell types (A, B):
1. Count how often cell type A has cell type B as a neighbor
2. Compare to a **random permutation null model**
3. Calculate a **z-score**: how many standard deviations away from random?

**Z-score interpretation:**
- **z > 2**: Cell types are **enriched** (occur together more than expected)
- **z < -2**: Cell types are **depleted** (avoid each other)
- **-2 < z < 2**: No significant spatial association

### Exercise 3.1: Compute Neighborhood Enrichment for One Image

In [ ]:
# Compute neighborhood enrichment
sq.gr.nhood_enrichment(
    adata_single, 
    cluster_key='celltypes',
    n_perms=1000,  # Number of random permutations for null model
    seed=42
)

print("Results stored in: adata_single.uns['celltypes_nhood_enrichment']")
print(f"Shape: {adata_single.uns['celltypes_nhood_enrichment']['zscore'].shape}")

In [ ]:
# Visualize enrichment as a heatmap
sq.pl.nhood_enrichment(
    adata_single,
    cluster_key='celltypes',
    method='average',
    cmap='coolwarm',
    vmin=-3,
    vmax=3,
    figsize=(10, 8)
)

plt.title(f'Neighborhood Enrichment (Image {IMAGE_ID})', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

**How to read this heatmap:**

- **Rows**: Cell type A ("I am this cell type")
- **Columns**: Cell type B ("My neighbor is this cell type")
- **Color**: Z-score of enrichment
  - **Red**: Enriched (positive z-score, more than expected)
  - **Blue**: Depleted (negative z-score, less than expected)
  - **White**: No association (z-score ≈ 0)

**Diagonal values** are always high (cells are neighbors of their own type).

### Exercise 3.2: Extract and Interpret Specific Interactions

In [ ]:
# Extract z-score matrix
zscore_matrix = adata_single.uns['celltypes_nhood_enrichment']['zscore']
cell_types = adata_single.obs['celltypes'].cat.categories

# Convert to DataFrame for easier inspection
zscore_df = pd.DataFrame(
    zscore_matrix,
    index=cell_types,
    columns=cell_types
)

print("Z-score matrix (rows: cell type, columns: neighbor type):")
print(zscore_df.round(2))

In [ ]:
# Find top 10 enriched interactions (excluding diagonal)
# Diagonal = cells being neighbors of themselves, always high

enriched_pairs = []

for i, ct1 in enumerate(cell_types):
    for j, ct2 in enumerate(cell_types):
        if i != j:  # Exclude diagonal
            enriched_pairs.append({
                'Cell_Type_A': ct1,
                'Neighbor_Type_B': ct2,
                'Z_Score': zscore_matrix[i, j]
            })

enriched_df = pd.DataFrame(enriched_pairs).sort_values('Z_Score', ascending=False)

print("\nTop 10 Enriched Interactions:")
print(enriched_df.head(10).to_string(index=False))

print("\nTop 10 Depleted Interactions:")
print(enriched_df.tail(10).to_string(index=False))

### Question 3.1:
**Look at the top enriched interactions. Do they make biological sense? For example:**
- Are immune cells enriched near tumor cells?
- Do stromal cell types cluster together?
- Are there any surprising enrichments or depletions?

*Write your interpretation below:*

**Your interpretation:**

- [Write here]

## 4. Neighborhood Analysis Across All Images

A single image might not be representative. Let's compute neighborhood enrichment **per image** and aggregate the results.

### Exercise 4.1: Compute Enrichment for All Images

In [ ]:
# This may take a few minutes
print("Computing spatial graphs for all images...")

# Build spatial graph for the full dataset, but do it per image
# We need to process each image separately
all_images = adata.obs['image'].unique()
print(f"Processing {len(all_images)} images...")

In [ ]:
# Store z-scores for each image
zscore_per_image = []
image_ids = []

for img_id in all_images[:20]:  # Process first 20 images for speed
    # Subset to this image
    adata_img = adata[adata.obs['image'] == img_id].copy()
    
    # Skip images with too few cells
    if adata_img.n_obs < 50:
        continue
    
    # Build spatial graph
    sq.gr.spatial_neighbors(
        adata_img,
        coord_type='generic',
        radius=20,
        spatial_key='spatial'
    )
    
    # Compute neighborhood enrichment
    try:
        sq.gr.nhood_enrichment(
            adata_img,
            cluster_key='celltypes',
            n_perms=500,  # Fewer permutations for speed
            seed=42
        )
        
        zscore_per_image.append(adata_img.uns['celltypes_nhood_enrichment']['zscore'])
        image_ids.append(img_id)
        
    except Exception as e:
        print(f"Skipping image {img_id}: {e}")
        continue

print(f"\nSuccessfully processed {len(zscore_per_image)} images")

### Exercise 4.2: Aggregate Results Across Images

In [ ]:
# Average z-scores across images
zscore_array = np.array(zscore_per_image)
zscore_mean = np.nanmean(zscore_array, axis=0)
zscore_std = np.nanstd(zscore_array, axis=0)

print(f"Average z-score matrix shape: {zscore_mean.shape}")
print(f"Processed {len(zscore_per_image)} images")

In [ ]:
# Visualize aggregated enrichment
cell_types = adata.obs['celltypes'].cat.categories
zscore_df_agg = pd.DataFrame(
    zscore_mean,
    index=cell_types,
    columns=cell_types
)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    zscore_df_agg,
    cmap='coolwarm',
    center=0,
    vmin=-3,
    vmax=3,
    square=True,
    linewidths=0.5,
    cbar_kws={'label': 'Mean Z-Score'},
    ax=ax
)

ax.set_title(f'Neighborhood Enrichment (Average across {len(zscore_per_image)} images)', 
             fontsize=14, pad=20)
ax.set_xlabel('Neighbor Cell Type', fontsize=12)
ax.set_ylabel('Center Cell Type', fontsize=12)

plt.tight_layout()
plt.show()

### Exercise 4.3: Identify Robust Spatial Associations

In [ ]:
# Find interactions that are consistently enriched across images
# We want high mean z-score AND low standard deviation

robust_pairs = []

for i, ct1 in enumerate(cell_types):
    for j, ct2 in enumerate(cell_types):
        if i != j:  # Exclude diagonal
            mean_z = zscore_mean[i, j]
            std_z = zscore_std[i, j]
            
            robust_pairs.append({
                'Cell_Type_A': ct1,
                'Neighbor_Type_B': ct2,
                'Mean_Z_Score': mean_z,
                'Std_Z_Score': std_z,
                'Coefficient_of_Variation': std_z / (abs(mean_z) + 1e-6)
            })

robust_df = pd.DataFrame(robust_pairs)

# Sort by mean z-score
robust_df_sorted = robust_df.sort_values('Mean_Z_Score', ascending=False)

print("Most Consistently Enriched Interactions:")
print(robust_df_sorted.head(15).to_string(index=False))

print("\nMost Consistently Depleted Interactions:")
print(robust_df_sorted.tail(15).to_string(index=False))

### Question 4.1:
**Compare the single-image results (Section 3) to the aggregated results (Section 4). Are there any differences? Which results are more reliable and why?**

**Your answer:**

- [Write here]

## 5. Biological Interpretation

Let's focus on biologically relevant interactions involving immune cells and tumor cells.

### Exercise 5.1: Tumor-Immune Interactions

In [ ]:
# Define immune and tumor cell types
immune_types = ['CD4_Tcell', 'CD8_Tcell', 'Treg', 'Bcell', 'Plasma_cell', 
                'NK', 'Macrophage', 'Monocyte', 'Neutrophil', 'DC']
tumor_types = ['Tumor']

# Filter for tumor-immune interactions
tumor_immune = robust_df[
    ((robust_df['Cell_Type_A'].isin(tumor_types)) & 
     (robust_df['Neighbor_Type_B'].isin(immune_types))) |
    ((robust_df['Cell_Type_A'].isin(immune_types)) & 
     (robust_df['Neighbor_Type_B'].isin(tumor_types)))
]

tumor_immune_sorted = tumor_immune.sort_values('Mean_Z_Score', ascending=False)

print("Tumor-Immune Cell Spatial Associations:")
print(tumor_immune_sorted.to_string(index=False))

In [ ]:
# Visualize tumor-immune interactions
fig, ax = plt.subplots(figsize=(10, 6))

# Create interaction labels
tumor_immune_sorted['Interaction'] = (
    tumor_immune_sorted['Cell_Type_A'] + ' ← ' + tumor_immune_sorted['Neighbor_Type_B']
)

# Bar plot
colors = ['red' if z > 0 else 'blue' for z in tumor_immune_sorted['Mean_Z_Score']]
ax.barh(tumor_immune_sorted['Interaction'], tumor_immune_sorted['Mean_Z_Score'], color=colors, alpha=0.7)

ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.axvline(2, color='gray', linewidth=0.5, linestyle=':', alpha=0.5)
ax.axvline(-2, color='gray', linewidth=0.5, linestyle=':', alpha=0.5)

ax.set_xlabel('Mean Z-Score (Enrichment →)', fontsize=12)
ax.set_title('Tumor-Immune Cell Spatial Associations', fontsize=14, pad=15)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

### Question 5.1:
**Which immune cell types are most enriched near tumor cells? What might this suggest about the immune response in these tumors?**

*Consider:*
- CD8 T cells (cytotoxic, can kill tumor cells)
- Tregs (regulatory, suppress immune response)
- Macrophages (can be pro- or anti-tumor)
- B cells (antibody production, tertiary lymphoid structures)

**Your interpretation:**

- [Write here]

### Exercise 5.2: Visualize a Specific Interaction in Tissue

In [ ]:
# Pick an image and visualize CD8 T cells near tumor cells
IMAGE_ID = 1
adata_vis = adata[adata.obs['image'] == IMAGE_ID].copy()

# Create a binary annotation: tumor vs CD8 vs other
def classify_cell(celltype):
    if celltype == 'Tumor':
        return 'Tumor'
    elif celltype == 'CD8_Tcell':
        return 'CD8 T cell'
    else:
        return 'Other'

adata_vis.obs['tumor_cd8'] = adata_vis.obs['celltypes'].apply(classify_cell)

# Plot
fig, ax = plt.subplots(1, 1, figsize=(12, 10))

coords = adata_vis.obsm['spatial']
cell_annotations = adata_vis.obs['tumor_cd8']

for cell_type, color in [('Other', 'lightgray'), ('Tumor', 'red'), ('CD8 T cell', 'blue')]:
    mask = cell_annotations == cell_type
    ax.scatter(
        coords[mask, 0],
        coords[mask, 1],
        c=color,
        s=10,
        alpha=0.6,
        label=cell_type
    )

ax.set_aspect('equal')
ax.legend(markerscale=2, loc='upper right')
ax.set_title(f'Spatial Distribution: Tumor Cells and CD8 T Cells (Image {IMAGE_ID})', fontsize=14)
ax.set_xlabel('X coordinate (μm)', fontsize=12)
ax.set_ylabel('Y coordinate (μm)', fontsize=12)

plt.tight_layout()
plt.show()

**Interpretation:**
- Look for regions where blue dots (CD8) are close to red dots (tumor)
- Are CD8 T cells infiltrating the tumor (mixed red and blue)?
- Or are they excluded (red and blue separated)?

## 6. Wrap-Up and Key Takeaways

### What You Learned:

1. **Spatial graphs** connect cells based on physical proximity
2. **Neighborhood enrichment** quantifies which cell types co-occur more or less than expected
3. **Z-scores** measure statistical significance of spatial associations
4. **Aggregating across images** provides more robust estimates than single-image analysis
5. **Biological interpretation** requires domain knowledge (e.g., CD8 T cells kill tumor cells)

### Next Steps:

In **Tutorial 4**, we will:
- Move beyond pairwise interactions to identify **cellular niches**
- Use CellCharter to discover multi-cell-type microenvironments
- Relate niche composition to clinical outcomes

---

### Optional Challenge:

**Try answering these questions with the code above:**

1. Which stromal cell types (fibroblast, endothelial) co-occur most strongly?
2. Do B cells and plasma cells cluster together (suggesting germinal centers)?
3. How does changing the radius parameter (10 μm vs 40 μm) affect enrichment results?